In [28]:
from pprint import pprint
from typing import OrderedDict

import torch
import torch.nn as nn

from src.modules import Transformer

In [36]:
def trainable_params(module: nn.Module):
    num_trainable_params = sum(
        [param.numel() for param in module.parameters() if param.requires_grad]
    )

    return num_trainable_params


def model_size(module: nn.Module):
    model_size = sum(
        [param.numel() * param.element_size() for param in module.parameters()]
    )

    return model_size


def flops_percentage(
    flops_comp: OrderedDict,
    num_layers: int,
    mha_keys: list = ["kqv", "qk_rope", "scores", "reduce", "proj"],
    mlp_keys: list = ["ffw1", "ffw2", "ffw3"],
):
    dict_percentages = OrderedDict()
    f_total = flops_comp["forward_total"]

    # Attention percentages
    for key in mha_keys:
        final_key = f"attention/{key}"
        total_flops_key = flops_comp[final_key] * num_layers

        dict_percentages[final_key] = (total_flops_key / f_total) * 100

    dict_percentages["attention"] = (
        (flops_comp["attention"] * num_layers) / f_total
    ) * 100

    # MLP percentages
    for key in mlp_keys:
        final_key = f"mlp/{key}"
        total_flops_key = flops_comp[final_key] * num_layers

        dict_percentages[final_key] = (total_flops_key / f_total) * 100

    dict_percentages["mlp"] = (
        (flops_comp["mlp"] * num_layers) / f_total
    ) * 100

    dict_percentages["transformer"] = (
        flops_comp["transformer"] / f_total
    ) * 100

    dict_percentages["dense"] = (flops_comp["dense"] / f_total) * 100

    return dict_percentages


def flops(
    num_layers: int = 12,
    context_length: int = 1024,
    d_model: int = 768,
    num_heads: int = 12,
    d_ff: int = 3072,
    vocab_size: int = 50257,
) -> OrderedDict[str, int]:
    # we only count Weight FLOPs, all other layers (LayerNorm, Softmax, etc) are effectively irrelevant
    # we count actual FLOPs, not MACs. Hence 2* all over the place
    # basically for any matrix multiply A (BxC) @ B (CxD) -> (BxD) flops are 2*B*C*D

    out = OrderedDict()
    head_size = d_model // num_heads

    # attention blocks
    # 1) the projection to key, query, values
    out["attention/kqv"] = (
        2 * context_length * (d_model * 3 * d_model)
    )  # CORRECT
    out["attention/qk_rope"] = int(2 * (2 * 2 * d_model / 2 * 2))
    # 2) calculating the attention scores
    out["attention/scores"] = (
        2 * context_length * context_length * d_model
    )  # CORRECT
    # 3) the reduction of the values (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
    out["attention/reduce"] = (
        2
        * num_heads
        * (context_length * context_length * head_size)  # CORRECT
    )
    # 4) the final linear projection
    out["attention/proj"] = 2 * context_length * (d_model * d_model)
    out["attention"] = sum(
        out["attention/" + k]
        for k in ["kqv", "qk_rope", "scores", "reduce", "proj"]
    )

    # MLP blocks
    out["mlp/ffw1"] = 2 * context_length * (d_model * d_ff)
    out["mlp/ffw2"] = 2 * context_length * (d_ff * d_model)
    out["mlp/ffw3"] = 2 * context_length * (d_model * d_ff)
    out["mlp"] = out["mlp/ffw1"] + out["mlp/ffw2"] + out["mlp/ffw3"]

    # the transformer and the rest of it
    out["block"] = out["attention"] + out["mlp"]
    out["transformer"] = num_layers * out["block"]
    out["dense"] = 2 * context_length * (d_model * vocab_size)

    # forward,backward,total
    out["forward_total"] = out["transformer"] + out["dense"]
    out["backward_total"] = (
        2 * out["forward_total"]
    )  # use common estimate of bwd = 2*fwd
    out["total"] = out["forward_total"] + out["backward_total"]

    return out


In [43]:
vocab_size = 50257
context_length = 1024
d_ff = 6400

In [44]:
# GPT-2 small
num_layers = 12
d_model = 768
num_heads = 12

flops_gpt_2_s = flops(
    num_layers, context_length, d_model, num_heads, d_ff, vocab_size
)

pprint(flops_gpt_2_s)

pprint(flops_percentage(flops_gpt_2_s, num_layers))

OrderedDict([('attention/kqv', 3623878656),
             ('attention/qk_rope', 6144),
             ('attention/scores', 1610612736),
             ('attention/reduce', 1610612736),
             ('attention/proj', 1207959552),
             ('attention', 8053069824),
             ('mlp/ffw1', 10066329600),
             ('mlp/ffw2', 10066329600),
             ('mlp/ffw3', 10066329600),
             ('mlp', 30198988800),
             ('block', 38252058624),
             ('transformer', 459024703488),
             ('dense', 79047426048),
             ('forward_total', 538072129536),
             ('backward_total', 1076144259072),
             ('total', 1614216388608)])
OrderedDict([('attention/kqv', 8.081917178929169),
             ('attention/qk_rope', 1.3702252161541698e-05),
             ('attention/scores', 3.591963190635187),
             ('attention/reduce', 3.591963190635187),
             ('attention/proj', 2.69397239297639),
             ('attention', 17.959829655428095),
          

In [45]:
# GPT-2 medium
num_layers = 24
d_model = 1024
num_heads = 16

flops_gpt_2_m = flops(
    num_layers, context_length, d_model, num_heads, d_ff, vocab_size
)

pprint(flops_gpt_2_m)

pprint(flops_percentage(flops_gpt_2_m, num_layers))

OrderedDict([('attention/kqv', 6442450944),
             ('attention/qk_rope', 8192),
             ('attention/scores', 2147483648),
             ('attention/reduce', 2147483648),
             ('attention/proj', 2147483648),
             ('attention', 12884910080),
             ('mlp/ffw1', 13421772800),
             ('mlp/ffw2', 13421772800),
             ('mlp/ffw3', 13421772800),
             ('mlp', 40265318400),
             ('block', 53150228480),
             ('transformer', 1275605483520),
             ('dense', 105396568064),
             ('forward_total', 1381002051584),
             ('backward_total', 2762004103168),
             ('total', 4143006154752)])
OrderedDict([('attention/kqv', 11.196132726859693),
             ('attention/qk_rope', 1.4236618966242082e-05),
             ('attention/scores', 3.7320442422865643),
             ('attention/reduce', 3.7320442422865643),
             ('attention/proj', 3.7320442422865643),
             ('attention', 22.392279690338352),
 

In [46]:
# GPT-2 big
num_layers = 36
d_model = 1280
num_heads = 20

flops_gpt_2_b = flops(
    num_layers, context_length, d_model, num_heads, d_ff, vocab_size
)

pprint(flops_gpt_2_b)

pprint(flops_percentage(flops_gpt_2_b, num_layers))

OrderedDict([('attention/kqv', 10066329600),
             ('attention/qk_rope', 10240),
             ('attention/scores', 2684354560),
             ('attention/reduce', 2684354560),
             ('attention/proj', 3355443200),
             ('attention', 18790492160),
             ('mlp/ffw1', 16777216000),
             ('mlp/ffw2', 16777216000),
             ('mlp/ffw3', 16777216000),
             ('mlp', 50331648000),
             ('block', 69122140160),
             ('transformer', 2488397045760),
             ('dense', 131745710080),
             ('forward_total', 2620142755840),
             ('backward_total', 5240285511680),
             ('total', 7860428267520)])
OrderedDict([('attention/kqv', 13.83084432297739),
             ('attention/qk_rope', 1.4069462405372509e-05),
             ('attention/scores', 3.688225152793971),
             ('attention/reduce', 3.688225152793971),
             ('attention/proj', 4.6102814409924635),
             ('attention', 25.8175901390202),
    

In [68]:
# GPT-2 XL
vocab_size = 50257
context_length = 1024

num_layers = 48
d_model = 1600
num_heads = 25

flops_gpt_2_xl = flops(
    num_layers, context_length, d_model, num_heads, d_ff, vocab_size
)

pprint(flops_gpt_2_xl)

pprint(flops_percentage(flops_gpt_2_xl, num_layers))

OrderedDict([('attention/kqv', 15728640000),
             ('attention/qk_rope', 12800),
             ('attention/scores', 3355443200),
             ('attention/reduce', 3355443200),
             ('attention/proj', 5242880000),
             ('attention', 27682419200),
             ('mlp/ffw1', 20971520000),
             ('mlp/ffw2', 20971520000),
             ('mlp/ffw3', 20971520000),
             ('mlp', 62914560000),
             ('block', 90596979200),
             ('transformer', 4348655001600),
             ('dense', 164682137600),
             ('forward_total', 4513337139200),
             ('backward_total', 9026674278400),
             ('total', 13540011417600)])
OrderedDict([('attention/kqv', 16.727638479358557),
             ('attention/qk_rope', 1.3612987043748828e-05),
             ('attention/scores', 3.5685628755964927),
             ('attention/reduce', 3.5685628755964927),
             ('attention/proj', 5.57587949311952),
             ('attention', 29.44065733665811),
 

In [71]:
# GPT-2 XL
vocab_size = 50257
context_length = 16384

num_layers = 48
d_model = 1600
num_heads = 25

flops_gpt_2_xl_cl = flops(
    num_layers, context_length, d_model, num_heads, d_ff, vocab_size
)

pprint(flops_gpt_2_xl_cl)

pprint(flops_percentage(flops_gpt_2_xl_cl, num_layers))

OrderedDict([('attention/kqv', 251658240000),
             ('attention/qk_rope', 12800),
             ('attention/scores', 858993459200),
             ('attention/reduce', 858993459200),
             ('attention/proj', 83886080000),
             ('attention', 2053531251200),
             ('mlp/ffw1', 335544320000),
             ('mlp/ffw2', 335544320000),
             ('mlp/ffw3', 335544320000),
             ('mlp', 1006632960000),
             ('block', 3060164211200),
             ('transformer', 146887882137600),
             ('dense', 2634914201600),
             ('forward_total', 149522796339200),
             ('backward_total', 299045592678400),
             ('total', 448568389017600)])
OrderedDict([('attention/kqv', 8.078765121939552),
             ('attention/qk_rope', 4.109072429371924e-07),
             ('attention/scores', 27.575518282887007),
             ('attention/reduce', 27.575518282887007),
             ('attention/proj', 2.692921707313184),
             ('attention',

In [1]:
flops_gpt_2_xl_cl["forward_total"] / flops_gpt_2_xl["forward_total"]

NameError: name 'flops_gpt_2_xl_cl' is not defined